In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pypulseq as pp
import MRzeroCore as mr0
import torchkbnufft as tkbn
from pypulseq import Sequence
from tqdm import tqdm
from typing import Optional, Tuple
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.FourierTransform import nonuniform_fourier_transform_forward, nonuniform_fourier_transform_adjoint
import Modules.ConcatSum 
from Modules.Regularizer import Regularizer
from Modules.CoilSens import cyclic_head_coil 
from Modules.Linops import LinearOperator, ChannelOperator, IdentityOperator
from Modules.NumSolvers import conjugate_gradient
import Modules.Resnet
from Modules.fast_iso import isochromat_sim_fast
torch.set_default_device("cuda")

In [ ]:
def main(plot: bool, write_seq: bool, seq_filename: str = "gre_radial_pypulseq.seq"):
    seq = pp.Sequence()  
    FOV = 260e-3
    N_x = 128 
    alpha = 5 
    R = 4
    slice_thickness = 3e-3  
    TE = 4e-3 
    TR = 100e-3  
    N_spokes = 256  
    N_dummy = 20  
    delta = np.pi / N_spokes  

    rf_spoiling_inc = 117 

    system = pp.Opts(
        max_grad=28,
        grad_unit="mT/m",
        max_slew=120,
        slew_unit="T/m/s",
        rf_ringdown_time=20e-6,
        rf_dead_time=100e-6,
        adc_dead_time=10e-6,
    )

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=alpha * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
    )

    deltak = 1 / FOV
    gx = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x, duration=gx.flat_time, delay=gx.rise_time, system=system
    )
    gx_pre = pp.make_trapezoid(
        channel="x", area=-gx.area / 2, duration=2e-3, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=2e-3, system=system
    )
    gx_spoil = pp.make_trapezoid(channel="x", area=0.5 * N_x * deltak, system=system)
    gz_spoil = pp.make_trapezoid(channel="z", area=4 / slice_thickness, system=system)
    
    delay_TE = (
        np.ceil(
            (
                TE
                - pp.calc_duration(gx_pre)
                - gz.fall_time
                - gz.flat_time / 2
                - pp.calc_duration(gx) / 2
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    delay_TR = (
        np.ceil(
            (
                TR
                - pp.calc_duration(gx_pre)
                - pp.calc_duration(gz)
                - pp.calc_duration(gx)
                - delay_TE
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    assert np.all(delay_TR) > pp.calc_duration(gx_spoil, gz_spoil)
    rf_phase = 0
    rf_inc = 0

    angles = np.linspace(0, np.pi, N_spokes // R , endpoint=False)

    for i in range(-N_dummy, N_spokes // R):

        if i < 0:
            angle = 0
        else:
            angle = angles[i]
    
        seq.add_block(rf, gz)
        seq.add_block(*pp.rotate(gx_pre, gz_reph, angle=angle, axis="z"))
        seq.add_block(pp.make_delay(delay_TE))
        if i > 0:
            seq.add_block(*pp.rotate(gx, adc, angle=angle, axis="z"))
        else:
            seq.add_block(*pp.rotate(gx, angle=angle, axis="z"))
        seq.add_block(
            *pp.rotate(gx_spoil, gz_spoil, pp.make_delay(delay_TR), angle=angle, axis="z")
        )

    ok, error_report = seq.check_timing()
    if ok:
        print("Timing check passed successfully")
    else:
        print("Timing check failed! Error listing follows:")
        print(error_report)

    if plot:
        seq.plot()

    if write_seq:
        seq.set_definition("FOV", [FOV, FOV, slice_thickness])
        seq.set_definition("Name", "gre_rad")
        seq.write("Radial 2D GRE")


if __name__ == "__main__":
    main(plot=True, write_seq=True)

In [ ]:
device = torch.device("cuda")
N_x = 128
N_channels = 8
FOV = 260e-3

phantom = mr0.util.load_phantom(size=(N_x, N_x))

coil_sensitivities_recon = cyclic_head_coil((N_channels, N_x, N_x, 1)).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (N_channels, N_x, N_x, 1),
    fov_scaling=(phantom.size[0] / FOV, phantom.size[1] / FOV, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

data = phantom.build()

In [ ]:
# Simulate the sequence (ground-truth time-domain spin simulation)
mr0_seq = mr0.Sequence.import_file("Radial 2D GRE.seq")
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr0_seq, data.cuda(), spin_count=256, spin_dist="rand", print_progress=True,
)
print(f"simulated signal: {tuple(signal.shape)}")

In [ ]:
class DeepADMM(nn.Module):

    def __init__(self, shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=5):

        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(shape)
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg

    def forward(self,Q,b,x, disable_progressbar=True):
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)

        for iteration in range(self.maxiter_admm):
    
            x = conjugate_gradient(
                Q + self.rho * self.I,
                b=b + self.rho * (z - u),
                x=x,
                maxiter=self.maxiter_cg,
                disable_progressbar=True,
            )
    
            z = self.regularizer(x + u)
            u = u + x - z
    
        return x

In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

In [ ]:
class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK = k.shape
        nC, nX, nY = input_shape

        self.n_modes = (nX, nY)
        self.k = k
        self.forward_shape = (nC, nX, nY, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

seq = pp.Sequence()
seq.read("Radial 2D GRE.seq")
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

F = NonuniformFourierTransformOperator(k_traj_adc, (N_channels, N_x, N_x))
C = ChannelOperator(coil_sensitivities_recon, (N_x, N_x))
A = F @ C

In [ ]:
data = signal.clone()
data = data.reshape((-1, N_channels)).permute((1, 0))
data = data.contiguous()
AHd = A.H @ data.view(torch.float32).ravel()
AHd_image = AHd.view(torch.complex64).reshape((N_x, N_x))
Q = A.H @ A
b = AHd
x = torch.zeros(N_x * N_x, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, b, x, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((N_x, N_x))

In [ ]:
plt.figure()
plt.imshow(torch.abs(x_recon_image).cpu().numpy().T, cmap="gray")
plt.show()

In [ ]:
torch.manual_seed(0)
shape = (N_x, N_x, 1)
model_init = DeepADMM(shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=5)

with torch.no_grad():
    x_deep_admm_init = model_init(Q, b, b)
x_deep_admm_init_image = x_deep_admm_init.view(torch.complex64).reshape((N_x, N_x))

In [ ]:
shape = (N_x, N_x, 1)

torch.manual_seed(0)
model = DeepADMM(
    shape,
    rho_init=0.1,
    maxiter_admm=10,
    maxiter_cg=5,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

nK = k_traj_adc.shape[1]
num_splits = 100

model.train()
losses = []

progress_bar = tqdm(range(num_splits), desc="Self-supervised splits")

for split in progress_bar:

    np.random.seed(split)

    indices = np.arange(nK)
    np.random.shuffle(indices)
    index_src, index_tar = np.split(indices, 2)

    k_traj_src = k_traj_adc[:, index_src]
    k_data_src = data[:, index_src].view(torch.float32).ravel()

    k_traj_tar = k_traj_adc[:, index_tar]
    k_data_tar = data[:, index_tar].view(torch.float32).ravel()

    F_src = NonuniformFourierTransformOperator(k_traj_src, (N_channels, N_x, N_x))
    F_tar = NonuniformFourierTransformOperator(k_traj_tar, (N_channels, N_x, N_x))
    C = ChannelOperator(coil_sensitivities_recon, (N_x, N_x))

    Q_src = C.H @ F_src.H @ F_src @ C
    b_src = C.H @ F_src.H @ k_data_src

    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)

    k_data_pred = F_tar @ C @ x_out

    loss = torch.linalg.norm(k_data_pred - k_data_tar, ord=1) / torch.linalg.norm(
        k_data_tar, ord=1
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

In [ ]:
model.eval()
F_full = NonuniformFourierTransformOperator(k_traj_adc, (N_channels, N_x, N_x))
C = ChannelOperator(coil_sensitivities_recon, (N_x, N_x))
x0 = torch.zeros(N_x * N_x, dtype=torch.complex64).view(torch.float32)
with torch.no_grad():
    x_deep_admm_trained = model(Q, b, b)

x_deep_admm_trained_image = x_deep_admm_trained.view(torch.complex64).reshape(
    (N_x, N_x)
)

In [ ]:
plt.imshow(torch.abs(x_deep_admm_trained_image).cpu().numpy().T, cmap="gray")
plt.title("DeepADMM (learned, trained)")
plt.show()

In [ ]:
plt.figure()
plt.plot(losses, marker="o")
plt.xlabel("Split")
plt.ylabel("Self-supervised loss")
plt.title("DeepADMM training loss over splits")
plt.show()